In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_1422_Dwarka-Sector_8_Delhi_DPCC__1Day.csv")

In [3]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,219.62,336.36,7.93,23.94,31.80,50.79,2.08,1.02,7.17,...,NaN,12.68,77.15,0.69,272.99,0.00,0.00,58.92,987.98,NaN
1,2024-01-02,209.98,312.71,14.77,24.43,38.90,48.36,2.91,0.92,8.85,...,NaN,12.08,75.58,0.65,256.04,0.00,0.00,63.27,987.31,NaN
2,2024-01-03,234.08,342.32,14.95,32.32,37.93,52.83,2.10,1.26,13.01,...,NaN,12.05,82.45,0.72,226.71,0.00,0.00,60.03,987.22,NaN
3,2024-01-04,234.11,345.63,28.85,25.35,36.95,46.18,7.29,1.33,9.30,...,NaN,11.56,86.89,0.96,262.21,0.00,0.00,43.32,987.56,NaN
4,2024-01-05,162.72,242.64,18.97,29.65,31.20,49.83,7.16,1.13,12.86,...,NaN,12.75,87.46,0.95,258.84,0.00,0.00,44.31,987.42,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,173.70,219.78,36.00,79.32,71.51,159.80,4.25,1.03,9.20,...,NaN,16.20,89.55,1.20,154.58,0.42,0.42,4.14,986.50,NaN
362,2024-12-28,99.50,119.82,38.05,63.59,64.75,104.03,6.20,1.83,5.06,...,NaN,16.35,91.94,0.88,223.59,0.01,0.01,9.82,986.22,NaN
363,2024-12-29,96.21,127.83,10.75,32.92,26.24,81.75,3.84,1.15,15.59,...,NaN,15.70,88.91,1.23,265.04,0.00,0.00,26.09,986.00,NaN
364,2024-12-30,100.83,147.38,29.24,40.68,45.44,70.53,3.73,1.44,20.83,...,NaN,14.50,86.31,1.08,258.95,0.00,0.00,29.64,986.00,NaN


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 21)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Xylene (µg/m³)']
Dropped rows (>70% NaN): 0
Missing values after imputation:
 Timestamp          0
PM2.5 (µg/m³)      0
PM10 (µg/m³)       0
NO (µg/m³)         0
NO2 (µg/m³)        0
NOx (ppb)          0
NH3 (µg/m³)        0
SO2 (µg/m³)        0
CO (mg/m³)         0
Ozone (µg/m³)      0
Benzene (µg/m³)    0
Toluene (µg/m³)    0
AT (°C)            0
RH (%)             0
WS (m/s)           0
WD (deg)           0
RF (mm)            0
TOT-RF (mm)        0
SR (W/mt2)         0
BP (mmHg)          0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (366, 20)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         219.62        336.36        7.93        23.94   
1  2024-01-02         209.98        312.71       14.77        24.43   
2  2024-01-03         234.08        342.32       14.95        32.32   
3  2024-01-04         234.11        345.63       28.85        25.35   
4  2024-01-05         162.72        242.64       18.97        29.65   

   NOx (ppb)  NH3 (µg/m³)  SO2 (µg/m³)  CO (mg/m³)  Ozone (µg/m³)  \
0      31.80        50.79         2.08        1.02           7.17   
1      38.90        48.36         2.91        0.92           8.85   
2      37.93        52.83         2.10        1.26          13.01   
3      36.95        46.18         7.29        1.33           9.30   
4      31.20        49.83         7.16        1.13          12.86   

   Benzene (µg/m³)  Toluene (µg/m³)  AT (°C)  RH (%)  WS (m/s)  WD (deg)  \
0             0.23             0.80    12.68   77.15      0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),Benzene (µg/m³),Toluene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg)
0,2024-01-01,1.685395,0.733988,-0.836993,-0.700382,-0.266258,0.916814,-2.085720,-0.750334,-1.835900,-0.749542,-1.042156,-1.743983,0.879639,-1.571406,1.584383,0.0,0.0,-1.007868,-0.296260
1,2024-01-02,1.549520,0.532686,-0.395689,-0.680543,0.001911,0.778045,-1.776445,-0.900503,-1.708717,-0.788079,-1.047190,-1.821349,0.780344,-1.744508,1.168695,0.0,0.0,-0.924812,-0.296260
2,2024-01-03,1.889208,0.784718,-0.384075,-0.361095,-0.034726,1.033311,-2.078268,-0.389927,-1.393786,-0.389863,-0.372585,-1.825218,1.214839,-1.441578,0.449397,0.0,0.0,-0.986674,-0.296260
3,2024-01-04,1.889631,0.812891,0.512728,-0.643294,-0.071741,0.653554,-0.144365,-0.284809,-1.674650,-0.094412,0.402708,-1.888400,1.495648,-0.402961,1.320011,0.0,0.0,-1.305724,-0.296260
4,2024-01-05,0.883392,-0.063730,-0.124712,-0.469197,-0.288920,0.861992,-0.192806,-0.585148,-1.405141,-0.261406,-0.146038,-1.734957,1.531698,-0.446237,1.237364,0.0,0.0,-1.286821,-0.296260
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,1.038155,-0.258308,0.974033,1.541831,1.233601,-0.208466,-1.277133,-0.735317,-1.682220,-0.210023,-0.090660,-1.290101,1.663881,0.635657,-1.319544,0.0,0.0,-2.053800,-0.296260
362,2024-12-28,-0.007691,-1.109139,1.106295,0.904958,0.978274,-0.208466,-0.550522,0.466038,-1.995637,1.832440,-0.090660,-1.270759,1.815037,-0.749167,0.372881,0.0,0.0,-1.945350,1.898702
363,2024-12-29,-0.054064,-1.040961,-0.655052,-0.336802,-0.476261,2.684825,-1.429907,-0.555114,-1.198468,0.830477,1.706610,-1.354573,1.623404,0.765484,1.389415,0.0,0.0,-1.634702,-0.515756
364,2024-12-30,0.011055,-0.874557,0.537890,-0.022617,0.248929,2.044092,-1.470896,-0.119623,-0.801776,0.920396,2.653071,-1.509305,1.458966,0.116348,1.240061,0.0,0.0,-1.566920,-0.515756


In [10]:
df.to_excel('Dwarkasector82024.xlsx', index=False)